# Lab06: Writing Data in Neo4j

Santiago Elí Jiménez Aguilar  
Luis Eduardo Gonzalez Gloria

## Goal:
Create a data pipeline to analyze the Recommendation Videogames dataset <br>
(https://networkrepository.com/rec-amz-Video-Games.php).

## Instructions
- **Dataset**. Download the Recommendation Videogames dataset from the Network Repository.
- **Data Ingestion**. This section should contain a code cell using PySpark to read the DataFrame.
- **Graph Analysis**. This section should contain the code to generate:
  - PageRank
  - Label Propagation
  - Triangle Counting
  - Degree Distribution
- **Writing Data in Neo4j**. This section should contain the code to persist the nodes and edges DataFrames in Neo4j.
- **Querying the Graph**. This section should contain a screenshot of the graph written to Neo4j.

## 2. Data Ingestion

In [1]:
from spark_utils import SparkUtils
neo4j_connector = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
su = SparkUtils("Lab06: Writing Data in Neo4j", "spark://spark-master:7077", spark_packages=neo4j_connector)
su.spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
io.graphframes#graphframes-spark3_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bff94d19-8781-4820-8bf9-c748f239cd9c;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.13;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.13_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in central
	found org.neo4j#caniuse-api;1.3.0 in central
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in central
	found org.jetbrains#annotations;13.0 in central
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in central
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in central
	found org.reactivestreams#reactiv

In [2]:
from graphframes import GraphFrame
from pyspark.sql import functions as F

# Define schema
mvideo_games_schema = SparkUtils.generate_schema([
    ("userId",      "string"),
    ("videoGameId", "string"),
    ("rating",      "float"),
    ("timestamp",   "long")
])

# Read the dataset
video_games_df = (su.spark.read
                .option("header", "false")
                .schema(mvideo_games_schema)
                .csv("/opt/spark/work-dir/data/rec-amz-Video-Games"))

video_games_df = video_games_df.filter(
    ~F.col("userId").startswith("<") &
    ~F.col("videoGameId").startswith("<")
)

# 1. User vertices
user_vertices = video_games_df.select(
    F.col("userId").alias("id"),
    F.lit("user").alias("type")
).distinct()

# 2. Game vertices
game_vertices = video_games_df.select(
    F.col("videoGameId").alias("id"),
    F.lit("game").alias("type")
).distinct()

# 3. Union both vertex types
vertices = user_vertices.union(game_vertices)

# 4. Edges
edges = video_games_df.select(
    F.col("userId").alias("src"),
    F.col("videoGameId").alias("dst"),
    F.col("rating").alias("rating"),
    F.col("timestamp").alias("timestamp")
)

# 5. Create the GraphFrame
g = GraphFrame(vertices, edges)
g.vertices.show()
g.edges.show()

/opt/spark/python/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
                                                                                

+--------------+----+
|            id|type|
+--------------+----+
|A228QXX35ISH3K|user|
|A2D33E7EBW17YD|user|
|A196O43KTFLJGK|user|
|A3C9HGVHE9I5VA|user|
|A12BT5SJ3N5H29|user|
|A125S8LNAPY2ND|user|
| AMBQ435QSSHFK|user|
|A37DLVPKJZGSQF|user|
|A3ER878Z9WVNRO|user|
|A2QDJZFRNSJNE3|user|
|A2L24HEU77W2JN|user|
|A1SYYW5IKU0MLU|user|
| A2D0A9VYN02RD|user|
| AZC7PNL9D7W8H|user|
|A3RQ4NTFLAQVJX|user|
|A1SQPEEHI5SPTR|user|
|A1A0SAY8C0ZK3K|user|
|A3TBF1VWVO690F|user|
|A2TPQAKCFO3VFX|user|
|A1S2LRS2MLPV2N|user|
+--------------+----+
only showing top 20 rows
+--------------+----------+------+----------+
|           src|       dst|rating| timestamp|
+--------------+----------+------+----------+
| AB9S9279OZ3QO|0078764343|   5.0|1373155200|
|A24SSUT5CSW8BH|0078764343|   5.0|1377302400|
| AK3V0HEBJMQ7J|0078764343|   4.0|1372896000|
|A10BECPH7W8HM7|043933702X|   5.0|1404950400|
|A2PRV9OULX1TWP|043933702X|   5.0|1386115200|
| AE7GUHCDQQ4UI|043933702X|   1.0|1366156800|
| A48ABFDDRMKI8|043933702X|   5.0

## 3. Graph Analysis

### PageRank

In [3]:
results = g.pageRank(resetProbability=0.15, maxIter=5)

print("=== TOP VIDEOJUEGOS POR PAGERANK ===")
results.vertices \
    .filter(F.col("type") == "game") \
    .select("id", "type", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(10)

/opt/spark/python/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


=== TOP VIDEOJUEGOS POR PAGERANK ===


+----------+----+------------------+
|        id|type|          pagerank|
+----------+----+------------------+
|B00DJFIMW6|game|7285.5264314177475|
|B00BGA9WK2|game| 2645.507195092002|
|B00FAX6XQC|game| 2547.334132845541|
|B009KS4XRO|game|2501.0385889521235|
|B0055SWM08|game| 1989.456325810203|
|B00CSR2J9I|game|1964.3732832793771|
|B002VBWIP6|game|1857.1934441905532|
|B0015AARJI|game|1408.3694893351437|
|B000FKBCX4|game|1243.5554664416438|
|B00178630A|game|1230.2746566197115|
+----------+----+------------------+
only showing top 10 rows


### Label Propagation

In [4]:
lpa = g.labelPropagation(maxIter=3)
lpa.show()

[Stage 389:====================================================>(198 + 1) / 200]

+--------------------+----+-------------+
|                  id|type|        label|
+--------------------+----+-------------+
|          0439394422|game|1202590846634|
|          9861064222|game|  60129543125|
|A08865193T9ASXJDB...|user|1657857380661|
|      A1016H15N3HBYY|user| 111669154003|
|       A102A7JAILVBB|user| 627065229369|
|      A106O5RPSDP4EH|user| 257698042096|
|      A108KKY6PMCEOT|user| 395136995536|
|      A10G7ODLW71IIP|user| 721554509924|
|      A10KHX41ONY4U1|user|1219770716319|
|      A10NK9KVHRMUXD|user|1709396988138|
|      A10R5R5CNTDYIA|user| 137438957682|
|      A10RFD2626P9JQ|user| 146028892443|
|      A10RISOZXLTTNM|user|1331439866057|
|      A10RT9F6D7CRE0|user|1443109015760|
|      A10RWD4STK4ZYM|user| 841813594254|
|      A10T5I8FGKEU71|user|1657857380630|
|      A10TC8V7K0UYN6|user| 506806145397|
|      A10TCHHRWQ7V58|user| 609885360293|
|      A10V3YWY35HDMJ|user|1460288884945|
|      A11158FHVYU0FL|user|         4366|
+--------------------+----+-------

### Triangle Counting

In [5]:
triangle_count = g.triangleCount()
triangle_count.show()

[Stage 514:======================================================>(66 + 1) / 67]

+-----+--------------+----+
|count|            id|type|
+-----+--------------+----+
|    0|A3TBF1VWVO690F|user|
|    0|A12BT5SJ3N5H29|user|
|    0|A196O43KTFLJGK|user|
|    0|A1SYYW5IKU0MLU|user|
|    0|A1A0SAY8C0ZK3K|user|
|    0|A2QDJZFRNSJNE3|user|
|    0|A2TPQAKCFO3VFX|user|
|    0|A1CMQW4D6JRXE3|user|
|    0|A3RQ4NTFLAQVJX|user|
|    0|A1SQPEEHI5SPTR|user|
|    0|A2D33E7EBW17YD|user|
|    0|A3ER878Z9WVNRO|user|
|    0|A1S2LRS2MLPV2N|user|
|    0|A3C9HGVHE9I5VA|user|
|    0|A228QXX35ISH3K|user|
|    0|A37DLVPKJZGSQF|user|
|    0| AMBQ435QSSHFK|user|
|    0| A2D0A9VYN02RD|user|
|    0|A2L24HEU77W2JN|user|
|    0|A125S8LNAPY2ND|user|
+-----+--------------+----+
only showing top 20 rows


### Degree Distribution

In [6]:
in_deg = g.inDegrees.join(vertices, "id")
in_deg.show()

+----------+--------+----+
|        id|inDegree|type|
+----------+--------+----+
|B00166PY5I|       7|game|
|B00009MGVG|       8|game|
|B001C4RAEW|       3|game|
|B001STDW2U|       4|game|
|B00008KTXO|       9|game|
|B001BP32TY|       5|game|
|B000046S3Z|      10|game|
|B0028SOQYS|       1|game|
|B003DU36N2|      18|game|
|B001E3WC44|      12|game|
|B000BUNE1G|       1|game|
|B00104KJ42|      71|game|
|B003FH3PE8|      24|game|
|B000MD51XQ|       2|game|
|B002BZ11E6|      33|game|
|B00006I5CP|       1|game|
|B0000AJVBW|     171|game|
|B001DW00YU|      13|game|
|B001TDLCRM|     148|game|
|B002EE1P2W|     121|game|
+----------+--------+----+
only showing top 20 rows


In [7]:
out_deg = g.outDegrees.join(vertices, "id")
out_deg.show()

[Stage 545:>                                                        (0 + 1) / 1]

+--------------------+---------+----+
|                  id|outDegree|type|
+--------------------+---------+----+
|A00010181745VTMHS...|        1|user|
|A0002090WKEMAO8KOWKM|        1|user|
|A00063061AK7XBIZL...|        1|user|
|A00065507CNSR8UHQFCK|        1|user|
|A0007530328GPY95A...|        1|user|
|A00082583JGF0RURT...|        1|user|
|A00089042GMZ1I1K4...|        1|user|
|A0009878M2RGMMHGJH39|        1|user|
|A00100742Q4O8VH0Y...|        1|user|
|A00101847G3FJTWYGNQA|        3|user|
|A00101961G0VS92WD...|        1|user|
|A0011102257KBXODK...|        1|user|
|A00116463B7B37ZYM...|        1|user|
|A00158021VAJ9275D...|        1|user|
|A00160082V0HQVAXB...|        1|user|
|A00190541RH03G6MZ...|        1|user|
|A001932810S6RCIFHJ3V|        1|user|
|A002080619B4767TS...|        1|user|
|A00230923E4Y7VHWZ...|        1|user|
|A002439424KGHR3LZ...|        1|user|
+--------------------+---------+----+
only showing top 20 rows


## 4. Writing Data in Neo4j

In [ ]:
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

# Write User nodes
g.vertices.filter(F.col("type") == "user").write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":User") \
  .option("node.keys", "id") \
  .save()

# Write Game nodes
g.vertices.filter(F.col("type") == "game").write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":Game") \
  .option("node.keys", "id") \
  .save()

# Write edges (User)-[:RATED]->(Game)
g.edges.repartition(8).write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("relationship", "RATED") \
  .option("relationship.save.strategy", "keys") \
  .option("relationship.source.labels", ":User") \
  .option("relationship.source.save.mode", "merge") \
  .option("relationship.source.node.keys", "src:id") \
  .option("relationship.target.labels", ":Game") \
  .option("relationship.target.save.mode", "merge") \
  .option("relationship.target.node.keys", "dst:id") \
  .option("batch.size", "5000") \
  .save()

print("Vertices and edges wrote in Neo4j")

In [ ]:
#su.spark.stop() 548